In [1]:
#pip install polytope


In [2]:
#pip install pypoman

In [3]:
# pip install polytope
# pip install pypoman

# import math
# import numpy as np
# from sympy.geometry import Point
# import matplotlib.pyplot as plt
# from polytope import polytope
# from polytope.polytope import enumerate_integral_points, qhull, box2poly
# from scipy.spatial import ConvexHull

Let's construct a reflexive 2D polytope (taken from the book "The Calabi–Yau Landscape", page 44)

In [4]:
#V = np.array([[0, -1], [-1, 0], [-1, 2], [1, 0], [1, -1]])


Pypoman provides a form $0\le -Ax+b$

In [5]:


# #print(compute_polytope_vertices(A, b))
# print(ConvexHull(V).npoints)

# GA TEST

In [6]:
# timeout to catch error of qhull, enumerate_integral_points, ABS_TOL infinity loop
# https://stackoverflow.com/questions/21827874/timeout-a-function-windows

from threading import Thread
import functools

#V= np.array([[-4, 3, 1, -3, 2], [1, 4, -4, -2, -3], [-1,  0,  2, -2,  1], [ 4,  0, -3, -2, -4], [ 2, -1,  0, -2,  1], [-1,  2,  0, -2,  1], [ 3, -3,  4,  1, -2]])

def timeout(timeout):
    def deco(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            res = [Exception('function [%s] timeout [%s seconds] exceeded!' % (func.__name__, timeout))]
            def newFunc():
                try:
                    res[0] = func(*args, **kwargs)
                except Exception as e:
                    res[0] = e
            t = Thread(target=newFunc)
            t.daemon = True
            try:
                t.start()
                t.join(timeout)
            except Exception as je:
                print ('error starting thread')
                raise je
            ret = res[0]
            if isinstance(ret, BaseException):
                raise ret
            return ret
        return wrapper
    return deco

In [7]:
import math
import numpy as np
from sympy.geometry import Point
import matplotlib.pyplot as plt
# from polytope import polytope
# from polytope.polytope import enumerate_integral_points, qhull, box2poly
from scipy.spatial import ConvexHull
import random
from random import randint
import copy
from fractions import Fraction
import cdd
from polytope.polytope import qhull, enumerate_integral_points, ABS_TOL
from itertools import combinations
import random
from numpy.random import choice










class Population:
    reduction_intervall = 10
    fitness_evolution = []
    def __init__(self, number_of_generations, survival_rate, crossover_type, crossover_parent_choosing_type, chrom_list, num_points_per_polytope, num_chroms_per_Generation, num_dimensions, min_value_per_point, max_value_per_point):
        #self.generation_0 = Generation(chrom_list, num_points_per_polytope, num_chroms_per_Generation, num_dimensions, min_value_per_point, max_value_per_point)
        #print(True)
        self.number_of_generations = number_of_generations
        self.all_generations = []
        


        self.all_generations.append(Generation(chrom_list, num_points_per_polytope, num_chroms_per_Generation, num_dimensions, min_value_per_point, max_value_per_point))
        #print(True)
        for curr_generation_number in range(0, number_of_generations-1):
            #print(True)
            if curr_generation_number % 100 == 0:
                print("curr Generation count: " + str(curr_generation_number))
                print(self.all_generations[curr_generation_number-1].get_chrom_list()[0].get_fitness())
                #print(repr(self.all_generations[curr_generation_number-1].get_chrom_list()[0].get_chromosom_string()))
            
            new_gen_list = []
            if curr_generation_number % (number_of_generations/Population.reduction_intervall) == 0 and curr_generation_number != 0:
                copy_prev_Generation_chrom_list = copy.deepcopy(self.all_generations[curr_generation_number].get_chrom_list())

                new_gen_list.append(copy.deepcopy(copy_prev_Generation_chrom_list[0]))
                #new_gen_list.append(copy.deepcopy(copy_prev_Generation_chrom_list[0]))
                for i in range(0, len(copy_prev_Generation_chrom_list)-1):
                    if copy_prev_Generation_chrom_list[i].get_chromosom_string() != copy_prev_Generation_chrom_list[i+1].get_chromosom_string():
                        new_gen_list.append(copy.deepcopy(copy_prev_Generation_chrom_list[i+1]))


                
                while len(new_gen_list) < num_chroms_per_Generation:
                    #chromosome_test_points=[]
                    polytope_points = []

                    for i in range(0,num_points_per_polytope):
                        polytope_points.append([random.randrange(min_value_per_point, max_value_per_point, 1) for i in range(num_dimensions)])#random.sample(range(min_value_per_point, max_value_per_point), num_dimensions))
                    
                    #polytope_points.sort(key=lambda x: int(x[0]))###########################################################################################
                    
                    chromosome_polytope_points = np.array(polytope_points)
                    #print(polytope_points)
                    chromosom_test = Chromosom(chromosome_polytope_points)

                    if chromosom_test.get_is_valid():
                        new_gen_list.append(chromosom_test)
                
                new_gen_list.sort(key=lambda x: x.fitness, reverse=True)
                
                print("REDUCTION PERFORMED!!!")
                
                #XXXXXX     .sort(key=lambda x: x.fitness, reverse=True)
                self.all_generations.append(Generation(new_gen_list, num_points_per_polytope, num_chroms_per_Generation, num_dimensions, min_value_per_point, max_value_per_point))


            else:    
                keeper_rate=0.1

                copy_prev_Generation_chrom_list = copy.deepcopy(self.all_generations[curr_generation_number].get_chrom_list())
                
                new_copy_prev_Generation_chrom_list = []
                for i in range(0,int(Generation.num_chroms_per_Generation*keeper_rate)):
                    new_copy_prev_Generation_chrom_list.append(copy_prev_Generation_chrom_list[i])
                

                fitness_list_of_prev_generation = self.all_generations[curr_generation_number].get_fitness_values_list()
                fitness_max = max(fitness_list_of_prev_generation)
                fitness_min = min(fitness_list_of_prev_generation) 
                
                #fitness_list_of_prev_generation = [((fitness_max-x)/fitness_max) for x in fitness_list_of_prev_generation]
                fitness_list_of_prev_generation = [Generation.normalize(min(fitness_list_of_prev_generation), max(fitness_list_of_prev_generation), i) for i in fitness_list_of_prev_generation]
               
                fitness_sum = sum(fitness_list_of_prev_generation)
                
                
                fitness_list_of_prev_generation = [Generation.adjust_fitness_values(i, fitness_sum) for i in fitness_list_of_prev_generation]
                
#                propabilities = list(range((len(copy_prev_Generation_chrom_list)*2), 0, -2))
#                sum_propabilities = sum(propabilities)
                #print(propabilities)
                #fitness_list_of_prev_generation = [Generation.adjust_fitness_values(i,sum_propabilities) for i in propabilities]
                #print(fitness_list_of_prev_generation)
                chromosomes_to_chose = int(int(Generation.num_chroms_per_Generation/2)-(Generation.num_chroms_per_Generation*keeper_rate))
                #print(propabilities)
                #copy_prev_Generation_chrom_list = random.choices()
                copy_prev_Generation_chrom_list = list(choice(copy_prev_Generation_chrom_list, size=chromosomes_to_chose,replace=False, p=fitness_list_of_prev_generation))

                copy_prev_Generation_chrom_list.extend(new_copy_prev_Generation_chrom_list)
                #print(len(copy_prev_Generation_chrom_list))
                # copy_prev_Generation_chrom_list = copy_prev_Generation_chrom_list[:int(len(copy_prev_Generation_chrom_list)/2)]
                
                #starting_length = len(self.all_generations[curr_generation_number].get_chrom_list())
                #parents_pool = list(combinations(copy_prev_Generation_chrom_list, 2))
                
                random.shuffle(copy_prev_Generation_chrom_list)
                #print(copy_prev_Generation_chrom_list)
                parents = copy_prev_Generation_chrom_list
                # parents1 = parents[:len(parents)//2]
                # parents2 = parents[len(parents)//2:]
                # parents_pool = list(zip(parents1, parents2))
                parents_pool = []
                for i in range(0, int(len(parents)/2)):
                    parents_tuple_list = [parents[i], parents[(len(parents)-i-1)]]
                    parents_pool.append(tuple(parents_tuple_list))







#################################################################                

                for parent_list_element in parents_pool:
                    #print(parent_list_element)
                    crossover_chromosom = Chromosom.apply_crossover(parent_list_element[0], parent_list_element[1], crossover_type)
                    if crossover_chromosom.get_is_valid():
                        new_gen_list.append(crossover_chromosom)


                #print(parents_pool)
###############################################################
                # #from multiprocessing import Pool
                # import multiprocessing

                # PROCESSES = 6
                # with multiprocessing.Pool(PROCESSES) as pool:
                #     params = parents_pool
                #     results = [pool.map(self.apply_crossover_parallel, p) for p in params]

                #print(results)
##

                # number_of_processes = 8
                # print(number_of_processes)
                # with Pool(number_of_processes) as p:
                #     results = p.map(self.apply_crossover_parallel, parents_pool)
                # print(results)



##

                # result = []
                # print(1)
                # pool = Pool(processes=2)
                # print(2)
                # res = pool.map_async(self.apply_crossover_parallel, parents_pool)
                # print(3)
                # result.append(res)
                # print(4)
                # print(res)
                # pool.close()
                # print(5)
                # pool.join()
                # print(result)
                # print(6)

################################################################
                # for i in results:
                #     if i.get().get_is_valid():
                #         print(i.get())
                #         new_gen_list.append(crossover_chromosom)



                parents = []
                while len(new_gen_list) < num_chroms_per_Generation/2: #len(self.all_generations[0].get_chrom_list()):
                    parents = random.sample(copy_prev_Generation_chrom_list, 2)
                    
                    crossover_chromosom = Chromosom.apply_crossover(parents[0], parents[1], crossover_type)
                    
                    if crossover_chromosom.get_is_valid():
                        new_gen_list.append(crossover_chromosom)
                    
                #try:
                num_to_mutate = random.randint(1,int(len(copy_prev_Generation_chrom_list)*0.3))

                number_sample_to_mutate = random.sample(range(0, len(copy_prev_Generation_chrom_list)), num_to_mutate)
                #copy_prev_Generation_chrom_list[:num_to_mutate]
                
                mut_counter = 0
                #mutate best two 
                mutated_chomosom1 = copy_prev_Generation_chrom_list[0].mutate_Chromosom()                
                mutated_chomosom2 = copy_prev_Generation_chrom_list[1].mutate_Chromosom()  

                # mutated_chomosom1 = mutated_chomosom1.mutate_Chromosom()
                # mutated_chomosom2 = mutated_chomosom2.mutate_Chromosom()

                if mutated_chomosom1.get_is_valid():
                    copy_prev_Generation_chrom_list.append(mutated_chomosom1)
                elif (mutated_chomosom2.get_is_valid()):   
                    copy_prev_Generation_chrom_list.append(mutated_chomosom2)
                
                mut_counter = 2

                for i in number_sample_to_mutate:
                    mutated_chomosom = copy_prev_Generation_chrom_list[i]
                    mutated_chomosom = mutated_chomosom.mutate_Chromosom()


                    #if mutated_chomosom.get_fitness() > copy_prev_Generation_chrom_list[i].get_fitness():
                    if mutated_chomosom.get_is_valid():
                        copy_prev_Generation_chrom_list.append(mutated_chomosom)
                    
                        mut_counter = mut_counter + 1



                # except:
                #     pass
                #print("next-Gen xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx")
                #print(starting_length)
                #print("new_gen_list")
                #print(len(new_gen_list))
                #print("copy_prev_Generation_chrom_list")
                #print(len(copy_prev_Generation_chrom_list))
                copy_prev_Generation_chrom_list.extend(new_gen_list)
                copy_prev_Generation_chrom_list.sort(key=lambda x: x.fitness, reverse=True)
                #print("mut_counter")
                #print(mut_counter)
                #print(len(copy_prev_Generation_chrom_list))
                copy_prev_Generation_chrom_list = copy_prev_Generation_chrom_list[:-mut_counter]
                #print(len(copy_prev_Generation_chrom_list))

                self.all_generations.append(Generation(copy_prev_Generation_chrom_list, num_points_per_polytope, num_chroms_per_Generation, num_dimensions, min_value_per_point, max_value_per_point))

    def apply_crossover_parallel(chrom_tuple):

        print("test")
        #new_chromosom = chrom_tuple
        print("crossover")
        chromosom_1 =chrom_tuple[0]
        chromosom_2 = chrom_tuple[1]
        crossover_type = 1# crossover_type

        new_chromosom_gen_list = []
        if crossover_type == 1: ##############to be removed


            if chromosom_1.get_fitness() > chromosom_2.get_fitness():
                chromosom_1_points = chromosom_1.get_gen_list()
                chromosom_2_points = chromosom_2.get_gen_list()
            else:
                chromosom_2_points = chromosom_1.get_gen_list()
                chromosom_1_points = chromosom_2.get_gen_list()

            
            if len(chromosom_1_points)>len(chromosom_2_points):
                length_new_chromosom = len(chromosom_2_points)
                random_crossover_point = random.randint(1,len(chromosom_2_points)-1)
            else:
                length_new_chromosom = len(chromosom_2_points)
                random_crossover_point = random.randint(1,len(chromosom_2_points)-1)

            new_gen = []
            
            
            for i in range(0,length_new_chromosom):
                
                if random.random() < 0.8:
                    if i < random_crossover_point:
                        new_gen = []
                        new_gen_2 = []
                        for j in chromosom_1_points[i]:
                            print(j)
                            if random.random() < 0.8:
                                new_gen.append(j)#[chromosom_1_points[0],chromosom_1_points[1]])
                            else:
                                j = randint(Generation.min_value_per_point, Generation.max_value_per_point)
                                new_gen.append(j)

                    else:
                        new_gen = []
                        for j in chromosom_2_points[i]:

                            
                            if random.random() < 0.8:
                                new_gen.append(j)#[chromosom_1_points[0],chromosom_1_points[1]])
                                
                            else:
                                j = randint(Generation.min_value_per_point, Generation.max_value_per_point)
                                new_gen.append(j)

                    new_chromosom_gen_list.append(new_gen)
                    #print("random_gen_test")
                else:
                    
                    new_chromosom_gen_list.append(list(chromosom_1_points[i]))
                    #print("random_gen_test")

        #new_chromosom_gen_list.sort(key=lambda x: int(x[0]))###################################################################################################################
            #print(new_chromosom_gen_list)
            new_chromosom = Chromosom(np.array(new_chromosom_gen_list))
            print(new_chromosom)

        return new_chromosom

    def get_all_generations(self):
        return self.all_generations


class Generation:
    num_points_per_polytope = 0
    num_chroms_per_Generation = 0
    num_dimensions = 0 # 3
    min_value_per_point = 0  # -3
    max_value_per_point = 0  # 3

    
    def __init__(self, chrom_list, num_points_per_polytope, num_chroms_per_Generation, num_dimensions, min_value_per_point, max_value_per_point):
        
        Generation.num_points_per_polytope = num_points_per_polytope
        Generation.num_chroms_per_Generation = num_chroms_per_Generation
        Generation.num_dimensions = num_dimensions # 3
        Generation.min_value_per_point = min_value_per_point  # -3
        Generation.max_value_per_point = max_value_per_point  # 3
        #print(1)
        if chrom_list == None:

            self.chrom_list = self.gen_first_generation(num_points_per_polytope, num_chroms_per_Generation, num_dimensions, min_value_per_point, max_value_per_point)
        else:
            self.chrom_list = chrom_list
        #print("two")
        self.generation_fitness_list = self.set_fitness_value_list(self.chrom_list)
        Population.fitness_evolution.append(self.generation_fitness_list[0])

    def set_fitness_value_list(self, chrom_list):
        fitness = []
        for chrom in chrom_list:
            fitness.append(chrom.get_fitness())

        #print(fitness)
        return(fitness)

    def get_chrom_list(self):
        return self.chrom_list
    
    def get_dimension():
        return Generation.num_dimensions

    def get_min_value_per_point():
        return Generation.min_value_per_point
    
    def get_max_value_per_point():
        return Generation.max_value_per_point

    def get_fitness_values_list(self):
        return self.generation_fitness_list

    def gen_first_generation(self, num_points_per_polytope, num_chroms_per_Generation, num_dimensions, min_value_per_point, max_value_per_point):
        #num_points_per_polytope = 5
        #num_chroms_per_Generation = 10
        #num_dimensions = num_dimensions

        polytope_points = []
        generation_polytopes = []
        counter = 0
        #for j in range(0, num_chroms_per_Generation):
        while (len(generation_polytopes)) < num_chroms_per_Generation:
            #chromosome_test_points=[]
            #print(len(generation_polytopes)+len(polytope_points))
            polytope_points = []
            
            # if len(generation_polytopes) == 0:
            #     polytope_points = [[0,-1,1,-1,1],[0,0,0,1,-2],[1,-1,0,0,0],[-1,1,1,1,0],[0,0,1,1,-1],[1,0,-1,-1,0],[0,1,0,0,0],[0,0,1,0,0],[0,0,-2,-1,0]]

            # # #     #polytope_points = [[-2, 3, 1, -3, 0], [1,-2,-1, 2, 0], [ 4,-4, 1, 2,-2], [ 1,2,3,0,-1], [-3,3,-1,-2,2], [ 1,2,3,-3,-1], [-2,-1,-3,0,2]]
            # # #     #polytope_points = [[-1,0,1,1,3],[3,4,-3,9,-5],[-1,-8,1,-15,3],[0,0,1,1,0],[-1,0,1,-7,3],[-1,-1,-1,-3,1]]
            # # #     polytope_points = [[  2,   0,  -2,   2,   3],[  1,  -2,  -2,  -4,   3],[ -4,  14,   1, -11,   0],[  0,   0,   0,   0,   0],[ 11,  -6,   0,  -2,   1],[  4,   1,  -9,  -8,  10],[  1, -12,   1, -12,   0],[  0,   0,  -1,  -6,   2],[ -7, -14,  -2,  -3,   3]]
            # elif(len(generation_polytopes) == 1):
            #     polytope_points = [[ -2, 0, 0, 1,-1], [1, 0, 0,-2,-2], [0, 0, 0, 1, 1], [ -2,-1, 0,-2,-1], [ -3, 2,-1, 5,-2], [1, 0, 0,-1, 0], [0,-1, 1, 1, 3], [6,-1, 1,-5, 1], [-12, 2,-2, 7,-7]]
            # # #     polytope_points = [[10,  18,  -2,  -9,   3], [  9,   4,  -2, -10,   3], [ 29,  28,   1, -10,   0], [  0,   0,   0,   0,   0,], [ -1,  -6,   0,  18,   1], [ 19,  19,  -9,  -4,  10], [ -4, -23,   1, -24,   0], [ 14,  11,  -1,  -7,   2], [-27,  29,  -2, -21,   3]]
            # # # # # #     polytope_points = [[-1,0,2,5,3],[3,4,-3,9,-5],[-1,-10,1,-7,3],[-1,0,0,-11,1],[-1,0,1,1,-1],[-1,0,1,-7,3]]
            # # # # # #     polytope_points = [[4, -2,  3,  2, -3], [ 1, -2, -1,  2,  0], [ 4, -4,  1,  2, -2], [ 1,  2,  3,  0, -1], [-3,  3, -1, -2,  2], [ 1,  2,  3, -3, -1,], [-2, -4, -5,  2,  1]]
            # # # # #     polytope_points = [[-1,0,0,0,-1],[1,0,1,-2,1],[-1,1,0,2,0],[-1,0,-1,3,0],[2,-1,1,-3,1],[-1,1,-1,2,-1],[-1,0,-1,2,0],[1,-1,0,-2,0],[-1,0,-1,1,-1]]

            # # elif(len(generation_polytopes) == 2):
            # #     polytope_points = [[ -2,   0,   0,   1,  -1],[  1,   0,   0,  -2,  -2],[  0,   0,   0,   1,   1],[ -2,  -2,   0,  -3,  -1],[ -3,   3,  -1,   6,  -2],[  1,   0,   0,  -1,   0],[  0,  -1,   1,   1,   3],[  6,  -1,   1,  -5,   1],[-12,   2,  -2,   7,  -7]]
            # # #     polytope_points = [[-1,0,0,0,-1],[1,0,1,0,1],[-1,1,0,2,0], [-1,-1,-1,-1,1], [-1,1,-1,2,-1],[1,-1,0,-1,0],[-1,0,-1,1,0],[2,-1,1,-2,1],[-1,1,0,1,-1]]

            # # # elif(len(generation_polytopes)==3):
            # # #     polytope_points = [[-1,  1,  0,  0,  0], [ 1, -1,  0, -1,  0],[-2,  0,  0,  1,  1],[ 0,  0,  0,  0, -1],[-3,  1, -1,  4, -2],[ 1,  0,  0, -1,  0],[ 0,  0,  0,  1,  1],[ 2, -2,  1, -4,  3],[-6,  2, -1,  6, -1]]

            # # # elif(len(generation_polytopes)==4):
            # # #     polytope_points = [[ -2, 0, 0, 1,-1], [1, 0, 0,-2,-2], [0, 0, 0, 1, 1], [ -2,-1, 0,-2,-1], [ -3, 2,-1, 5,-2], [1, 0, 0,-1, 0], [0,-1, 1, 1, 3], [6,-1, 1,-5, 1], [-12, 2,-2, 7,-7]]

            # else:

            counter += 1
            for i in range(0,num_points_per_polytope):
                polytope_points.append([random.randrange(min_value_per_point, max_value_per_point, 1) for i in range(num_dimensions)])#random.sample(range(min_value_per_point, max_value_per_point), num_dimensions))

            #print(polytope_points)
                
            #polytope_points.sort(key=lambda x: int(x[0])) ################################################################################
            
            chromosome_polytope_points = np.array(polytope_points)
            #print(polytope_points)
            chromosom_test = Chromosom(chromosome_polytope_points)
            #print(chromosom_test.get_is_valid())
            if chromosom_test.get_is_valid():
                generation_polytopes.append(chromosom_test)
        
        generation_polytopes.sort(key=lambda x: x.fitness, reverse=True)
        #print("generation done")
        return generation_polytopes
    
    def normalize(min, max, value):
        
        #new_value = (max-value)/max
        
        normalized_value = (value-min)/(max-min)
        return normalized_value

    def adjust_fitness_values(fitness_value, total_sum):
        propability = (fitness_value / total_sum)

        return float(propability)

class Chromosom:
    chrom_count = 0
    reflexive_politopes = {}
    all_chromosomes = {}

    def __init__(self, points):
        chromosome_polytope_points = points
        #print("pre try")
        #print(Generation.num_dimensions)
        self.is_valid = False
        self.chromosom_string = np.array2string(chromosome_polytope_points).replace("\n", ",")

        if self.chromosom_string in Chromosom.all_chromosomes.keys():
            self.fitness = Chromosom.all_chromosomes[self.chromosom_string]
            Chromosom.chrom_count = Chromosom.chrom_count + 1
            self.gen_list = chromosome_polytope_points
            self.is_valid = True
        else:
            #result = None
            #while result is None:
            try:
                internal_points = []
                #print("###############################################")
                Chromosom.chrom_count = Chromosom.chrom_count + 1
                #print(Chromosom.chrom_count)
                
                self.gen_list = chromosome_polytope_points
                self.fitness = self.calc_fitness(chromosome_polytope_points)
                #print(True)
                self.is_valid = True


                if self.fitness == 0 and self.chromosom_string not in Chromosom.reflexive_politopes.keys(): 
                    print(True)
                    print(repr(self.gen_list))
                    #if np.min(self.gen_list) < 0:
                    self.fitness = np.max(self.gen_list) - np.min(self.gen_list)
                    
                    
                    #self.fitness = np.sum(np.absolute(self.gen_list))
                    
                    
                        #self.fitness = np.max(self.gen_list) + (np.min(self.gen_list) *-1)
                    # elif np.min(self.gen_list) > 0:
                    #     self.fitness = np.max(self.gen_list) - np.min(self.gen_list)   
                    # elif np.min(self.gen_list)==0:
                    #     self.fitness = np.max(self.gen_list)
                    #polytope_volume = ConvexHull(chromosome_polytope_points).volume #qhull(chromosome_polytope_points).volume
                    #Chromosom.reflexive_politopes[polytope_volume] = [ConvexHull(self.gen_list), self.gen_list]
                    Chromosom.reflexive_politopes[self.chromosom_string] = [repr(ConvexHull(self.gen_list)), self.gen_list]
                    #print(Chromosom.reflexive_politopes.keys())
                        
                    Chromosom.all_chromosomes[self.chromosom_string] = self.fitness

                #result = 1
            except:
                self.is_valid = False
    
    def get_chromosom_string(self):
        return self.chromosom_string

    def get_is_valid(self):
        return self.is_valid
    
    def get_gen_list(self):
        return self.gen_list
    
    def get_fitness(self):
        return self.fitness

    def get_gen_list(self):
        return self.gen_list






    #code based on https://github.com/stephane-caron/pypoman/blob/master/pypoman/duality.py
    def compute_distances(self, vertices: np.ndarray) -> np.ndarray:
        #print("yyyyyyyyyyyyyyyyyyyyyyyyyyyyy")
        
        V = np.vstack(vertices)
        t = np.ones((V.shape[0], 1))  # first column is 1 for vertices
        tV = np.hstack([t, V])
        mat = cdd.Matrix(tV, number_type="fraction")
        mat.rep_type = cdd.RepType.GENERATOR
        P = cdd.Polyhedron(mat)
        bA = np.array(P.get_inequalities())
        if bA.shape == (0,):  # bA == []
            return bA
        # the polyhedron is given by b + A x >= 0 where bA = [b|A]
        b = bA[:, 0]
        A = -np.array(bA[:, 1:])

        for i, row in enumerate(A):
            denominators = []
            has_fraction = False
            for j in row:
                if type(j) == Fraction:
                    has_fraction = True
                    denominators.append(j.denominator)    
            if has_fraction:
                b[i] *= np.lcm.reduce(denominators)
        return b

    @timeout(10)
    def calc_fitness(self, vertices):
        
        # integral_points = enumerate_integral_points(qhull(vertices))
        # integral_points = integral_points.transpose()
        # integral_points = integral_points.astype(int)
        # ip_count = len(integral_points)
        # print("xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx")
        # print(np.array(vertices))
        distances = self.compute_distances(np.array(vertices))
        result = 0
        # if ip_count > 1:
        #     result -= 1
        for d in distances:
            result -= abs(d-1)
        
        return int(result)

    def apply_crossover(chromosom_1, chromosom_2, crossover_type):
        #print("crossover")
        # chromosom_1 =
        # chromosom_2
        crossover_type = crossover_type

        new_chromosom_gen_list = []
        if crossover_type == 1: ##############to be removed

            if chromosom_1.get_fitness() > chromosom_2.get_fitness():
                chromosom_1_points = chromosom_1.get_gen_list()
                chromosom_2_points = chromosom_2.get_gen_list()
            else:
                chromosom_2_points = chromosom_1.get_gen_list()
                chromosom_1_points = chromosom_2.get_gen_list()

            
            if len(chromosom_1_points)>len(chromosom_2_points):
                length_new_chromosom = len(chromosom_2_points)
                random_crossover_point = random.randint(1,len(chromosom_2_points)-1)
            else:
                length_new_chromosom = len(chromosom_2_points)
                random_crossover_point = random.randint(1,len(chromosom_2_points)-1)

            #new_gen = []
            
            
            for i in range(0,length_new_chromosom):
                
                if random.random() < 0.8:
                    if i < random_crossover_point:
                        new_gen = []
                        new_gen_2 = []
                        for j in chromosom_1_points[i]:
                            
                            if random.random() < 0.8:
                                new_gen.append(j)#[chromosom_1_points[0],chromosom_1_points[1]])
                            else:
                                j = randint(Generation.min_value_per_point, Generation.max_value_per_point)
                                new_gen.append(j)

                    else:
                        new_gen = []
                        for j in chromosom_2_points[i]:

                            
                            if random.random() < 0.8:
                                new_gen.append(j)#[chromosom_1_points[0],chromosom_1_points[1]])
                            else:
                                j = randint(Generation.min_value_per_point, Generation.max_value_per_point)
                                new_gen.append(j)

                    new_chromosom_gen_list.append(new_gen)
                    #print("random_gen_test")
                else:
                    #for i in range(0,Generation.num_points_per_polytope):
                    new_chromosom_gen_list.append([random.randrange(Generation.min_value_per_point, Generation.max_value_per_point, 1) for i in range(Generation.num_dimensions)])#random.sample(range(Generation.min_value_per_point, Generation.max_value_per_point), Generation.num_dimensions))
                    #print(new_chromosom_gen_list[len(new_chromosom_gen_list)-1])
                    #print("random_gen_test")

        #new_chromosom_gen_list.sort(key=lambda x: int(x[0]))###################################################################################################################

            new_chromosom = Chromosom(np.array(new_chromosom_gen_list))

            return new_chromosom

    # def apply_crossover_parallel(chrom_tuple):
    #     print("test")
    #     #new_chromosom = chrom_tuple
    #     print("crossover")
    #     chromosom_1 =chrom_tuple[0]
    #     chromosom_2 = chrom_tuple[1]
    #     crossover_type = 1# crossover_type

    #     new_chromosom_gen_list = []
    #     if crossover_type == 1: ##############to be removed

    #         if chromosom_1.get_fitness() > chromosom_2.get_fitness():
    #             chromosom_1_points = chromosom_1.get_gen_list()
    #             chromosom_2_points = chromosom_2.get_gen_list()
    #         else:
    #             chromosom_2_points = chromosom_1.get_gen_list()
    #             chromosom_1_points = chromosom_2.get_gen_list()

            
    #         if len(chromosom_1_points)>len(chromosom_2_points):
    #             length_new_chromosom = len(chromosom_2_points)
    #             random_crossover_point = random.randint(1,len(chromosom_2_points)-1)
    #         else:
    #             length_new_chromosom = len(chromosom_2_points)
    #             random_crossover_point = random.randint(1,len(chromosom_2_points)-1)

    #         #new_gen = []
            
            
    #         for i in range(0,length_new_chromosom):
                
    #             if random.random() < 0.8:
    #                 if i < random_crossover_point:
    #                     new_gen = []
    #                     new_gen_2 = []
    #                     for j in chromosom_1_points[i]:
    #                         print(j)
    #                         if random.random() < 0.8:
    #                             new_gen.append(j)#[chromosom_1_points[0],chromosom_1_points[1]])
    #                         else:
    #                             j = randint(Generation.min_value_per_point, Generation.max_value_per_point)
    #                             new_gen.append(j)

    #                 else:
    #                     new_gen = []
    #                     for j in chromosom_2_points[i]:

                            
    #                         if random.random() < 0.8:
    #                             new_gen.append(j)#[chromosom_1_points[0],chromosom_1_points[1]])
                                
    #                         else:
    #                             j = randint(Generation.min_value_per_point, Generation.max_value_per_point)
    #                             new_gen.append(j)

    #                 new_chromosom_gen_list.append(new_gen)
    #                 #print("random_gen_test")
    #             else:
                    
    #                 new_chromosom_gen_list.append(random.sample(range(Generation.min_value_per_point, Generation.max_value_per_point), Generation.num_dimensions))
    #                 #print("random_gen_test")

    #     #new_chromosom_gen_list.sort(key=lambda x: int(x[0]))###################################################################################################################
    #         #print(new_chromosom_gen_list)
    #         new_chromosom = Chromosom(np.array(new_chromosom_gen_list))
    #         print(new_chromosom)

    #     return new_chromosom





    def mut_one(gen_list):

        chrom_points = gen_list
        random_point = random.randint(0 ,Generation.num_points_per_polytope)
        random_axis = random.randint(0,Generation.num_dimensions)
        random_value = random.random()

        if random_value < 0.1:
            chrom_points[random_point-1][random_axis-1] = random.randint(Generation.min_value_per_point, Generation.max_value_per_point) #(-3, 3)
        elif random_value > 0.55:
        #if random.random() > 0.5:
            chrom_points[random_point-1][random_axis-1] = chrom_points[random_point-1][random_axis-1] -1 #random.randint(Generation.min_value_per_point, Generation.max_value_per_point) #(-3, 3)
        else:
            chrom_points[random_point-1][random_axis-1] = chrom_points[random_point-1][random_axis-1] +1

        return chrom_points



    def mut_two(gen_list):
        random_value = random.random()
        chrom_points = gen_list
        random_point = random.randint(0 ,Generation.num_points_per_polytope)
        random_axis = random.randint(0,Generation.num_dimensions)
        # if random.random()<0.1:
        #     chrom_points[random_point-1][random_axis-1] = random.randint(Generation.min_value_per_point, Generation.max_value_per_point) #(-3, 3)
        #elif random.random() > 0.55:
        if random_value > 0.5:
            chrom_points[random_point-1][random_axis-1] = chrom_points[random_point-1][random_axis-1] -1 #random.randint(Generation.min_value_per_point, Generation.max_value_per_point) #(-3, 3)
        else:
            chrom_points[random_point-1][random_axis-1] = chrom_points[random_point-1][random_axis-1] +1

        return chrom_points


    def mut_three(gen_list):
        random_value = random.random()
        chrom_points = gen_list
        random_point = random.randint(0 ,Generation.num_points_per_polytope)
        random_axis = random.randint(0,Generation.num_dimensions)
        # if random.random()<0.1:
        #     chrom_points[random_point-1][random_axis-1] = random.randint(Generation.min_value_per_point, Generation.max_value_per_point) #(-3, 3)
        #elif random.random() > 0.55:
        if random_value <= 0.25:
            chrom_points[random_point-1][random_axis-1] = chrom_points[random_point-1][random_axis-1] -1 #random.randint(Generation.min_value_per_point, Generation.max_value_per_point) #(-3, 3)
        elif random_value > 0.25 and random_value<= 0.5:
            chrom_points[random_point-1][random_axis-1] = chrom_points[random_point-1][random_axis-1] +1
        elif random_value > 0.5 and random_value<= 0.75:
            chrom_points[random_point-1][random_axis-1] = chrom_points[random_point-1][random_axis-1] -2

        elif random_value > 0.75:
            chrom_points[random_point-1][random_axis-1] = chrom_points[random_point-1][random_axis-1] +2

        return chrom_points




    def mutate_Chromosom(self):
        
        chrom_points = copy.deepcopy(self.get_gen_list())
        
        #chrom_points[random_point-1][random_axis-1] = random.randint(Generation.min_value_per_point, Generation.max_value_per_point) #(-3, 3)
        
        for i in range(0,random.randint(1,Generation.num_points_per_polytope)):
            # random_point = random.randint(0 ,Generation.num_points_per_polytope)
            # random_axis = random.randint(0,Generation.num_dimensions)
            # # if random.random()<0.1:
            # #     chrom_points[random_point-1][random_axis-1] = random.randint(Generation.min_value_per_point, Generation.max_value_per_point) #(-3, 3)
            # #elif random.random() > 0.55:
            # if random.random() > 0.5:
            #     chrom_points[random_point-1][random_axis-1] = chrom_points[random_point-1][random_axis-1] -1 #random.randint(Generation.min_value_per_point, Generation.max_value_per_point) #(-3, 3)
            # else:
            #     chrom_points[random_point-1][random_axis-1] = chrom_points[random_point-1][random_axis-1] +1
            chrom_points = Chromosom.mut_one(chrom_points)



        new_chromosom = Chromosom(chrom_points)
        #print(new_chromosom.get_chromosom_string())
        #print("done")
        return new_chromosom

    #def apply_crossover_return_2_children(chromosom_1, chromosom_2, crossover_type):



In [8]:
# Population Test:

# test_population = Population(number_of_generations, survival_rate, crossover_type, crossover_parent_choosing_type, chrom_list, num_points_per_polytope, num_chroms_per_Generation, num_dimensions, min_value_per_point, max_value_per_point)
Population.reduction_intervall = 5 # Generations / reduction intervall

number_of_generations = 10000
survival_rate = None
crossover_type = 1
crossover_parent_choosing_type = None
chrom_list = None
num_points_per_polytope = 13
num_chroms_per_Generation = 50
num_dimensions = 5
min_value_per_point = -2
max_value_per_point = 2

test_population = Population(number_of_generations, survival_rate, crossover_type, crossover_parent_choosing_type, chrom_list, num_points_per_polytope, num_chroms_per_Generation, num_dimensions, min_value_per_point, max_value_per_point)



curr Generation count: 0
-423
curr Generation count: 100
-25
curr Generation count: 200
-3
curr Generation count: 300
-1
True
array([[ 0,  0,  0,  1,  1],
       [ 1,  0, -2,  0,  0],
       [ 1,  1, -2, -1, -1],
       [ 1,  1, -1, -2, -1],
       [ 1,  0, -1,  0,  0],
       [-1,  0,  1, -1,  0],
       [ 0,  0, -1,  0,  0],
       [ 1, -1,  0, -1, -1],
       [ 1,  1, -1, -2, -1],
       [ 1,  0, -1, -1,  0],
       [-1,  1,  1, -2, -1],
       [ 1, -1,  0, -1,  0],
       [ 1, -1,  0, -1, -3]])
curr Generation count: 400
4
True
array([[ 0,  0,  0,  1,  1],
       [ 1,  0, -2,  0,  0],
       [ 1,  1, -2, -1, -1],
       [ 1,  1, -1, -2, -1],
       [ 1,  0, -1,  0,  0],
       [-1,  0,  1, -1,  0],
       [ 0,  0, -1,  0,  0],
       [ 1, -1,  0, -1, -1],
       [ 1,  1, -1, -2, -1],
       [ 1,  0, -1, -1,  0],
       [-1,  1,  1, -2, -1],
       [ 1, -1,  0, -1,  1],
       [ 1, -1,  0, -1, -3]])
True
array([[ 0,  0,  0,  1,  1],
       [ 1,  0, -2,  0,  0],
       [ 1,  1, -2, -

In [9]:

# for i in Chromosom.reflexive_politopes:
#     print(i)
#     print(Chromosom.reflexive_politopes[i][1])

print(Chromosom.reflexive_politopes)
print(len(Chromosom.reflexive_politopes))

{'[[ 0  0  0  1  1], [ 1  0 -2  0  0], [ 1  1 -2 -1 -1], [ 1  1 -1 -2 -1], [ 1  0 -1  0  0], [-1  0  1 -1  0], [ 0  0 -1  0  0], [ 1 -1  0 -1 -1], [ 1  1 -1 -2 -1], [ 1  0 -1 -1  0], [-1  1  1 -2 -1], [ 1 -1  0 -1  0], [ 1 -1  0 -1 -3]]': ['<scipy.spatial._qhull.ConvexHull object at 0x0000022B06179690>', array([[ 0,  0,  0,  1,  1],
       [ 1,  0, -2,  0,  0],
       [ 1,  1, -2, -1, -1],
       [ 1,  1, -1, -2, -1],
       [ 1,  0, -1,  0,  0],
       [-1,  0,  1, -1,  0],
       [ 0,  0, -1,  0,  0],
       [ 1, -1,  0, -1, -1],
       [ 1,  1, -1, -2, -1],
       [ 1,  0, -1, -1,  0],
       [-1,  1,  1, -2, -1],
       [ 1, -1,  0, -1,  0],
       [ 1, -1,  0, -1, -3]])], '[[ 0  0  0  1  1], [ 1  0 -2  0  0], [ 1  1 -2 -1 -1], [ 1  1 -1 -2 -1], [ 1  0 -1  0  0], [-1  0  1 -1  0], [ 0  0 -1  0  0], [ 1 -1  0 -1 -1], [ 1  1 -1 -2 -1], [ 1  0 -1 -1  0], [-1  1  1 -2 -1], [ 1 -1  0 -1  1], [ 1 -1  0 -1 -3]]': ['<scipy.spatial._qhull.ConvexHull object at 0x0000022B0621B690>', array([[ 

In [10]:
# V = np.array([[1, 0, -2], [-2, -3, 2], [-1, -2, 1], [1, 2, -1], [1, 2, 0]])

# print(Chromosom.calc_fitness(V))

In [11]:
test_population_generations = test_population.get_all_generations()
test_population_generations_chroms = test_population_generations[1].get_chrom_list()
#print(test_population_generations_chroms)

all_fitness = []
for generation in test_population.get_all_generations():
    #fitness = [generation.get_fitness_values_list()]
    # for chrom in generation.get_chrom_list():
    #     fitness.append(chrom.get_fitness())

    print(generation.get_fitness_values_list())

    all_fitness.append(generation.get_fitness_values_list())
    #print(generation.get_fitness_values_list())
    #all_fitness.append(fitness)
#print(all_fitness)
print(len(all_fitness))
#print(all_fitness)
print(Chromosom.reflexive_politopes)
print(len(Chromosom.reflexive_politopes))

[-423, -458, -464, -467, -467, -497, -512, -530, -537, -556, -556, -559, -578, -609, -620, -650, -691, -692, -705, -735, -750, -793, -800, -802, -808, -844, -854, -886, -890, -900, -905, -937, -970, -1004, -1030, -1047, -1058, -1127, -1129, -1132, -1140, -1148, -1150, -1234, -1261, -1261, -1266, -1393, -1909, -2228]
[-382, -423, -423, -434, -458, -464, -467, -467, -467, -477, -512, -537, -542, -556, -559, -578, -598, -600, -609, -620, -642, -650, -691, -705, -732, -735, -740, -750, -808, -839, -875, -883, -883, -886, -905, -965, -992, -993, -1030, -1041, -1048, -1061, -1123, -1140, -1150, -1204, -1208, -1221, -1234]
[-382, -382, -423, -423, -423, -423, -434, -458, -458, -467, -504, -512, -537, -559, -598, -600, -608, -609, -619, -620, -622, -626, -640, -642, -650, -662, -704, -750, -802, -802, -808, -823, -860, -886, -890, -910, -925, -944, -965, -984, -1000, -1020, -1059, -1061, -1109, -1123, -1142, -1167, -1180]
[-285, -332, -382, -382, -423, -423, -423, -423, -423, -434, -458, -458,

In [12]:
# test_population_generations = test_population.get_all_generations()
# test_population_generations_chroms = test_population_generations[2].get_chrom_list()
# #print(test_population_generations_chroms)

# print(test_population_generations_chroms[0].get_gen_list())
# print(test_population_generations_chroms[0].get_fitness())

# # all_fitness = []
# # for generation in test_population.get_all_generations():
# #     fitness = [generation.get_fitness_values_list()]
# #     for chrom in generation.get_chrom_list():
# #         fitness.append(chrom.get_fitness())

# #     #print(generation.get_fitness_values_list())

# #     all_fitness.append(generation.get_fitness_values_list())
# #     #print(generation.get_fitness_values_list())
# #     #all_fitness.append(fitness)
# # #print(all_fitness)
# # print(len(all_fitness))
# # #print(all_fitness)

In [13]:
number_of_generations = 100000

chrom_list = test_population_generations[len(test_population.get_all_generations())-1].get_chrom_list()

test_population2 = Population(number_of_generations, survival_rate, crossover_type, crossover_parent_choosing_type, chrom_list, num_points_per_polytope, num_chroms_per_Generation, num_dimensions, min_value_per_point, max_value_per_point)

curr Generation count: 0
6
True
array([[ 0,  0,  0,  1,  1],
       [ 1,  0, -2,  0,  0],
       [ 1,  1, -2, -1, -1],
       [ 1,  1, -1, -2, -1],
       [ 1,  0, -1,  0,  0],
       [-1,  0,  1, -1,  0],
       [ 0,  0,  0,  0,  0],
       [ 1, -1,  0, -1, -2],
       [ 1,  1, -1, -2, -1],
       [ 1,  0, -1,  0, -1],
       [-1,  1,  1, -2, -1],
       [ 1, -1,  0, -1,  1],
       [ 1, -1,  0, -1, -5]])
curr Generation count: 100
6
True
array([[ 0,  0,  0,  1,  1],
       [ 1,  0, -2,  0,  0],
       [ 1,  1, -2, -1, -1],
       [ 1,  1, -1, -2, -1],
       [ 1,  0, -2,  0,  0],
       [-1,  0,  1, -1,  0],
       [ 0,  0,  0,  0,  0],
       [ 1, -1,  0, -1, -2],
       [ 1,  1, -1, -2,  1],
       [ 1,  0, -1, -1, -1],
       [-1,  1,  1, -2, -1],
       [ 1, -1,  0, -1, -1],
       [ 1, -1,  0, -1, -5]])
curr Generation count: 200
6
curr Generation count: 300
6
curr Generation count: 400
6
True
array([[ 0,  0,  0,  1,  1],
       [ 1,  0, -2,  0, -1],
       [ 1,  1, -2, -1, -2],

In [14]:
test_population_generations2 = test_population2.get_all_generations()
test_population_generations_chroms2 = test_population_generations2[1].get_chrom_list()
#print(test_population_generations_chroms)

all_fitness = []
for generation in test_population2.get_all_generations():
    #fitness = [generation.get_fitness_values_list()]
    # for chrom in generation.get_chrom_list():
    #     fitness.append(chrom.get_fitness())

    print(generation.get_fitness_values_list())

    all_fitness.append(generation.get_fitness_values_list())
    #print(generation.get_fitness_values_list())
    #all_fitness.append(fitness)
#print(all_fitness)
print(len(all_fitness))
#print(all_fitness)
print(Chromosom.reflexive_politopes)
print(len(Chromosom.reflexive_politopes))

[6, 6, 6, 6, 6, 6, 6, 6, 6, 6, -25, -40, -51, -144, -146, -154, -155, -165, -244, -256, -263, -265, -295, -296, -299, -322, -324, -353, -389, -397, -428, -439, -461, -500, -511, -519, -522, -529, -532, -585, -607, -621, -710, -848, -857, -952, -963, -975, -995]
[6, 6, 6, 6, 6, 6, 6, 6, 6, 6, -25, -32, -40, -51, -115, -121, -146, -154, -244, -265, -296, -299, -304, -321, -322, -383, -391, -436, -451, -483, -492, -511, -519, -524, -585, -585, -588, -592, -606, -669, -700, -710, -753, -763, -786, -843, -854, -857, -1205]
[6, 6, 6, 6, 6, 6, 6, 6, 6, -121, -122, -146, -154, -244, -265, -277, -278, -296, -304, -308, -321, -336, -340, -377, -378, -381, -383, -397, -400, -451, -457, -519, -524, -531, -534, -585, -586, -616, -700, -710, -727, -741, -750, -787, -841, -841, -857, -870, -966]
[6, 6, 6, 6, 6, 6, 6, 6, 6, 6, -36, -53, -121, -146, -154, -202, -244, -278, -296, -304, -307, -336, -367, -377, -383, -387, -403, -451, -457, -530, -531, -534, -585, -586, -596, -750, -765, -830, -834, -846,

In [15]:
test_population_generations_chroms5 = test_population_generations2[499].get_chrom_list()
print(repr(test_population_generations_chroms5[0].get_gen_list()))

array([[ 0,  0,  0,  1,  1],
       [ 1,  0, -2,  0,  0],
       [ 1,  1, -2, -1, -1],
       [ 1,  1, -1, -2, -1],
       [ 1,  0, -1,  0,  0],
       [-1,  0,  1, -1,  0],
       [ 0,  0,  0,  0,  0],
       [ 1, -1,  0, -1, -2],
       [ 1,  1, -1, -2, -1],
       [ 1,  0, -1,  0, -1],
       [-1,  1,  1, -2, -1],
       [ 1, -1,  0, -1, -1],
       [ 1, -1,  0, -1, -5]])


In [16]:
chrom_list = test_population_generations2[len(test_population2.get_all_generations())-1].get_chrom_list()

test_population3 = Population(number_of_generations, survival_rate, crossover_type, crossover_parent_choosing_type, chrom_list, num_points_per_polytope, num_chroms_per_Generation, num_dimensions, min_value_per_point, max_value_per_point)

curr Generation count: 0
9
True
array([[ 0,  0,  0,  1,  1],
       [ 1,  0, -2,  0,  0],
       [ 1,  1, -2, -1,  0],
       [ 0,  1,  0, -2, -1],
       [ 1,  0, -1,  0, -2],
       [-1,  0,  1, -1, -1],
       [ 0,  0,  0,  0, -1],
       [ 1, -1,  0, -1, -6],
       [ 0,  1,  0, -2, -2],
       [ 1,  0, -1,  0,  1],
       [-1,  1,  1, -2, -1],
       [ 1, -1,  0, -1, -2],
       [ 1, -1,  0, -1, -3]])
curr Generation count: 100
9
True
array([[-1,  0,  1,  1,  3],
       [ 1,  0, -2,  0,  0],
       [ 1,  1, -2, -1,  0],
       [ 0,  1,  0, -2, -1],
       [ 1,  0, -1,  0, -1],
       [-1,  0,  1, -1, -1],
       [ 0,  0,  0,  0, -1],
       [ 1, -1,  0, -1, -6],
       [ 0,  1,  0, -2, -2],
       [ 1,  0, -1,  0,  1],
       [-1,  1,  1, -2, -1],
       [ 1, -1,  0, -1, -2],
       [ 1, -1,  0, -1, -3]])
curr Generation count: 200
9
curr Generation count: 300
9
curr Generation count: 400
9
True
array([[ 0,  0,  0,  1,  1],
       [ 1,  0, -2,  0,  0],
       [ 1,  1, -2, -1,  0],

In [17]:
test_population_generations3 = test_population3.get_all_generations()
#test_population_generations_chroms2 = test_population_generations2[999].get_chrom_list()
#print(test_population_generations_chroms)

all_fitness = []
for generation in test_population3.get_all_generations():
    #fitness = [generation.get_fitness_values_list()]
    # for chrom in generation.get_chrom_list():
    #     fitness.append(chrom.get_fitness())

    print(generation.get_fitness_values_list())

    all_fitness.append(generation.get_fitness_values_list())
    #print(generation.get_fitness_values_list())
    #all_fitness.append(fitness)
#print(all_fitness)
print(len(all_fitness))
#print(all_fitness)
print(Chromosom.reflexive_politopes)
print(len(Chromosom.reflexive_politopes))

[9, 9, 9, 9, 9, 9, 9, 9, 9, 9, -88, -113, -139, -167, -174, -192, -266, -288, -315, -351, -359, -360, -360, -374, -438, -457, -476, -521, -538, -566, -594, -594, -612, -623, -626, -629, -699, -705, -711, -735, -771, -774, -781, -826, -868, -907, -925, -936, -1006]
[9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, -139, -167, -174, -264, -266, -380, -438, -451, -457, -457, -478, -509, -538, -539, -550, -556, -566, -573, -583, -594, -594, -623, -630, -646, -697, -705, -826, -840, -952, -1010, -1042, -1059, -1135, -1215, -1225, -1258, -1295]
[9, 9, 9, 9, 9, 9, 9, 9, 9, 9, -167, -174, -208, -223, -300, -457, -474, -476, -484, -538, -539, -550, -556, -566, -573, -582, -583, -594, -630, -646, -648, -651, -724, -791, -800, -826, -872, -954, -999, -1059, -1089, -1159, -1160, -1205, -1296, -1463, -1476, -1503, -1569]
[9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, -165, -167, -174, -175, -208, -230, -358, -365, -375, -415, -450, -457, -519, -556, -564, -566, -583, -594, -646, -648, -791, -831, -837, -908, -921, -963, 

In [18]:
chrom_list = test_population_generations3[len(test_population3.get_all_generations())-1].get_chrom_list()

test_population4 = Population(number_of_generations, survival_rate, crossover_type, crossover_parent_choosing_type, chrom_list, num_points_per_polytope, num_chroms_per_Generation, num_dimensions, min_value_per_point, max_value_per_point)

curr Generation count: 0
9
curr Generation count: 100
9
curr Generation count: 200
9
curr Generation count: 300
9
curr Generation count: 400
9
curr Generation count: 500
9
curr Generation count: 600
9
curr Generation count: 700
9
curr Generation count: 800
9
True
array([[ 0,  0,  0,  1,  3],
       [ 1,  0, -2,  0,  0],
       [ 1,  1, -2, -1,  0],
       [ 0,  1,  0, -2, -1],
       [ 1,  0, -1,  0,  1],
       [-1,  0,  1, -1, -1],
       [ 0,  0,  0,  0, -1],
       [ 1, -1,  0, -1, -6],
       [ 0,  1,  0, -2, -1],
       [ 1,  0, -1,  0,  1],
       [-1,  1,  1, -2, -1],
       [ 1, -1,  0, -1, -3],
       [ 1, -1,  0, -1, -2]])
curr Generation count: 900
9
curr Generation count: 1000
9
True
array([[ 0,  0,  0,  1,  3],
       [ 1,  0, -2,  0,  0],
       [ 1,  1, -2, -1,  0],
       [ 0,  1,  0, -2, -1],
       [ 0,  0,  0,  0,  1],
       [-1,  0,  1, -1, -1],
       [ 0,  0,  0,  0, -1],
       [ 1, -1,  0, -1, -6],
       [ 0,  1,  0, -2, -1],
       [ 1,  0, -1,  0,  1],
    

In [19]:
test_population_generations4 = test_population4.get_all_generations()
#test_population_generations_chroms2 = test_population_generations2[999].get_chrom_list()
#print(test_population_generations_chroms)

all_fitness = []
for generation in test_population4.get_all_generations():
    #fitness = [generation.get_fitness_values_list()]
    # for chrom in generation.get_chrom_list():
    #     fitness.append(chrom.get_fitness())

    print(generation.get_fitness_values_list())

    all_fitness.append(generation.get_fitness_values_list())
    #print(generation.get_fitness_values_list())
    #all_fitness.append(fitness)
#print(all_fitness)
print(len(all_fitness))

print(Chromosom.reflexive_politopes)
print(len(Chromosom.reflexive_politopes))

[9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, -10, -40, -45, -51, -77, -86, -93, -108, -130, -141, -176, -181, -278, -292, -307, -318, -334, -347, -400, -428, -436, -562, -600, -603, -642, -647, -647, -692, -747, -770, -784, -829, -926, -1027, -1151, -1218, -1344]
[9, 9, 9, 9, 9, 9, 9, 9, 9, -40, -45, -46, -51, -54, -93, -99, -108, -130, -141, -143, -157, -181, -292, -307, -412, -425, -428, -436, -469, -510, -520, -524, -647, -663, -692, -696, -699, -784, -834, -837, -913, -926, -929, -941, -944, -974, -1000, -1011, -1016]
[9, 9, 9, 9, 9, 9, 9, 9, 9, -45, -46, -51, -54, -96, -108, -130, -141, -163, -181, -303, -307, -328, -341, -372, -386, -404, -412, -428, -436, -493, -510, -524, -563, -647, -647, -663, -689, -791, -803, -816, -848, -852, -860, -904, -965, -983, -1001, -1043, -1193]
[9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, -99, -108, -130, -141, -163, -171, -189, -303, -331, -403, -411, -412, -436, -475, -493, -510, -524, -537, -539, -563, -600, -646, -646, -663, -689, -791, -840, -864, -873, -884, -

In [ ]:
chrom_list = test_population_generations4[len(test_population4.get_all_generations())-1].get_chrom_list()

#number_of_generations = 200


test_population5 = Population(number_of_generations, survival_rate, crossover_type, crossover_parent_choosing_type, chrom_list, num_points_per_polytope, num_chroms_per_Generation, num_dimensions, min_value_per_point, max_value_per_point)



In [ ]:
test_population_generations5 = test_population5.get_all_generations()
#test_population_generations_chroms2 = test_population_generations2[999].get_chrom_list()
#print(test_population_generations_chroms)

all_fitness = []
for generation in test_population5.get_all_generations():
    #fitness = [generation.get_fitness_values_list()]
    # for chrom in generation.get_chrom_list():
    #     fitness.append(chrom.get_fitness())

    print(generation.get_fitness_values_list())

    all_fitness.append(generation.get_fitness_values_list())
    #print(generation.get_fitness_values_list())
    #all_fitness.append(fitness)
#print(all_fitness)
print(len(all_fitness))
print(Chromosom.reflexive_politopes)
print(len(Chromosom.reflexive_politopes))

In [ ]:
test_population_generations_chroms5 = test_population_generations2[99999].get_chrom_list()
print(repr(test_population_generations_chroms5[1].get_gen_list()))


# fitness -5 : array([[  2,   0,  -2,   2,   3],
#        [  1,  -2,  -2,  -4,   3],
#        [ -4,  14,   1, -11,   0],
#        [  0,   0,   0,   0,   0],
#        [ 11,  -6,   0,  -2,   1],
#        [  4,   1,  -9,  -8,  10],
#        [  1, -12,   1, -12,   0],
#        [  0,   0,  -1,  -6,   2],
#        [ -7, -14,  -2,  -3,   3]])

In [ ]:


# a = [1,2,3]
# b = a.copy()
# #b = []
# #b.append(a[0])
# #b.append(a[1])
# a[1] = 5

# print(a)
# print(b)

for i in Chromosom.reflexive_politopes:
    print(i)
    print(Chromosom.reflexive_politopes[i][1])

print(Chromosom.reflexive_politopes)
print(len(Chromosom.reflexive_politopes))

In [ ]:
# Crossover test - to be checked

# # #def apply_crossover(chromosom_1, chromosom_2, crossover_type):
# chromosom_1_points = test_polytopes[1].get_gen_list()
# chromosom_2_points = test_polytopes[0].get_gen_list()
# # crossover_type = 1

# # new_chromosom_gen_list = []
# # if crossover_type == 1:
    

# #     if len(chromosom_1_points)>len(chromosom_2_points):
# #         length_new_chromosom = len(chromosom_2_points)
# #         random_crossover_point = random.randint(1,len(chromosom_2_points)-1)
# #     else:
# #         length_new_chromosom = len(chromosom_2_points)
# #         random_crossover_point = random.randint(1,len(chromosom_2_points)-1)

# #     #new_gen = []
# #     for i in range(0,length_new_chromosom):
# #         if i <= random_crossover_point:
# #             new_gen = []
# #             for j in chromosom_1_points[i]:
# #                 new_gen.append(j)#[chromosom_1_points[0],chromosom_1_points[1]])
# #         else:
# #             new_gen = []
# #             for j in chromosom_2_points[i]:
# #                 new_gen.append(j)

# #         new_chromosom_gen_list.append(new_gen)

# # new_chromosom = Chromosom(np.array(new_chromosom_gen_list))
# #return new_chromosom

# new_chromosom = Chromosom.apply_crossover(test_polytopes[1], test_polytopes[0], 1)

# #print(random_crossover_point)
# print(chromosom_1_points)
# print(chromosom_2_points)
# print(new_chromosom.get_gen_list())
# print(new_chromosom.get_fitness())


In [ ]:
# import numpy as np
# import polytope

# # halfspace representation computed using `polytope`
# # from the vertex representation given in the question
# vertices = np.array([[0, -1], [-1, 0], [-1, 2], [1, 0], [1, -1]])
# poly = polytope.qhull(vertices)
# # first halfspace representation
# A = np.array([
#     [0, -1],
#     [-1, 0],
#     [-1, -1],
#     [1, 1],
#     [1, 0]])
# b = np.array([1, 1, 1, 1, 1])
# question_poly_1 = polytope.Polytope(A, b)
# # second halfspace representation
# A = np.array([
#     [-0.70711, -0.70711],
#     [-1, -0],
#     [0.70711, 0.70711],
#     [0, -1],
#     [1, 0]])
# b = np.array([0.70711, 1, 0.70711, 1, 1])

# question_poly_2 = polytope.Polytope(A, b)
# # check that all the above halfspace representations
# # represent the same polytope
# assert poly == question_poly_1, (poly, question_poly_1)
# assert poly == question_poly_2, (poly, question_poly_2)

In [ ]:
# inner_point = np.array([0,0]) # in this case it is the origin
# fac1 = polytope.quickhull.Facet([[0, -1], [-1, 0]])
# fac2 = polytope.quickhull.Facet([[-1, 2], [1, 0]])
# fac3 = polytope.quickhull.Facet([[1, -1], [0, -1]])

# d1 = polytope.quickhull.distance(inner_point, fac1)
# d2 = polytope.quickhull.distance(inner_point, fac2)
# d3 = polytope.quickhull.distance(inner_point, fac3)
# print(d1)
# print(d2)
# print(d3)

Let's visualize our 2D polytope. For this purpose we use the [Polytope Library](https://github.com/tulip-control/polytope)

In [ ]:
# project into 2D



In [ ]:

# print(polytope.polytope.cheby_ball(P))

# min_V = np.min(V, axis=0) #min val of each cols
# min_x = min_V[0]
# min_y = min_V[1]
# max_V = np.max(V, axis=0) #max val of each cols
# max_x = max_V[0]
# max_y = max_V[1]

# ax = P.plot(linestyle="solid", linewidth=1, color="lightblue", edgecolor="red")
# ax.set_xlim([min_x-1, max_x+1])
# ax.set_ylim([min_y-1, max_y+1])
# plt.show()